# ECR cost-comment parsing with an LLM

The notebook is for choosing the model, data, analysis instructions, and prompt. [`ecr_topics/llm_caller.py`](../ecr_topics/llm_caller.py) sends the request to the OpenAI-compatible endpoint configured in `.env` (e.g. the U-M GPT Toolkit).

Run the setup cell, choose the model and column, review/edit the instructions and prompt, then run the analysis cell. The initial test parses 20 nonblank `COST_COMMENT` values and saves a CSV.

## Setup and selections

Run the setup cells once, choose a model and workbook column, then run the analysis cell. The analysis instructions and prompt are ordinary Python strings that you can edit.

In [ ]:
%pip install -q openai python-dotenv ipywidgets pandas openpyxl python-docx

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from docx import Document
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from ecr_topics.llm_caller import available_models, call_llm, default_model

DATA_FILE = ROOT / "data" / "ECR Analytics Project Data 1.xlsx"
PROCESS_FLOW_FILE = ROOT / "data" / "Overall Process Flow.docx"
SHEET_NAME = "result"
if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Workbook not found: {DATA_FILE.resolve()}")
if not PROCESS_FLOW_FILE.is_file():
    raise FileNotFoundError(f"Process guide not found: {PROCESS_FLOW_FILE.resolve()}")

ecr_data = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)
if "SUPPLIER_CONTACT_NAME" in ecr_data.columns:
    ecr_data = ecr_data.drop(columns="SUPPLIER_CONTACT_NAME")

models = available_models()
if not models:
    raise RuntimeError("The LLM endpoint returned no models for this account.")
configured_model = default_model()
model_selector = widgets.Dropdown(
    options=models,
    value=configured_model if configured_model in models else models[0],
    description="Model:",
    layout=widgets.Layout(width="70%"),
)
column_selector = widgets.Dropdown(
    options=list(ecr_data.columns),
    value="COST_COMMENT" if "COST_COMMENT" in ecr_data.columns else ecr_data.columns[0],
    description="Column:",
    layout=widgets.Layout(width="70%"),
)
display(model_selector, column_selector)

process_flow = Document(PROCESS_FLOW_FILE)
flow_text = [p.text.strip() for p in process_flow.paragraphs if p.text.strip()]
flow_text.extend(
    " | ".join(cell.text.strip().replace("\n", " / ") for cell in row.cells)
    for table in process_flow.tables
    for row in table.rows
)
PROCESS_FLOW_CONTEXT = "\n".join(flow_text)

ANALYSIS_INSTRUCTIONS = """You extract structured cost information from ECR Cost Comment text. The process-flow guide defines Cost Comment as free-text context explaining a cost entry, such as a cost increase or decrease, supplier quote details, tooling assumptions, or other cost-specific context. Use only explicit evidence in each comment. Never infer an FX rate, convert currencies, or invent an amount. Use null when a requested value is absent or ambiguous. Do not include personal contact details in the reason."""

ANALYSIS_PROMPT = """For every supplied comment, extract exactly: total_cost_usd (explicit total in USD as a number, otherwise null), fx_rate (explicit exchange-rate value as written, otherwise null), and reason (concise stated reason for the cost, otherwise null). Do not calculate or convert costs or combine line items unless the comment explicitly states a total. Return only a JSON object with a records array. Each record must contain comment_id, total_cost_usd, fx_rate, and reason; preserve each supplied comment_id exactly once."""
NUMBER_OF_ROWS = 20

## Analyze and save

This sends only the selected column’s first 20 nonblank values (or fewer if configured) plus the process-flow guide to the caller. The output CSV contains the source text and the three extracted fields.

In [ ]:
selected_column = column_selector.value
selected_values = ecr_data[selected_column].dropna().map(str).str.strip()
selected_values = selected_values[selected_values.ne("")].head(NUMBER_OF_ROWS)
if selected_values.empty:
    raise ValueError(f"Column {selected_column!r} has no nonblank values to analyze.")

input_rows = [
    {"comment_id": row_id, "comment": value}
    for row_id, value in enumerate(selected_values, start=1)
]
result = call_llm(
    model=model_selector.value,
    instructions=f"{ANALYSIS_INSTRUCTIONS}\n\nECR process-flow guide:\n{PROCESS_FLOW_CONTEXT}",
    prompt=ANALYSIS_PROMPT,
    data=input_rows,
)

records = result.get("records")
expected_ids = set(range(1, len(input_rows) + 1))
if not isinstance(records, list) or any(not isinstance(row, dict) for row in records):
    raise ValueError("The model response must contain a records array of JSON objects.")
if len(records) != len(input_rows) or {row.get("comment_id") for row in records} != expected_ids:
    raise ValueError("The model must return exactly one parsed result per input value.")
required_fields = {"total_cost_usd", "fx_rate", "reason"}
if any(not required_fields.issubset(row) for row in records):
    raise ValueError("A model result is missing total_cost_usd, fx_rate, or reason.")

source_column = "cost_comment" if selected_column == "COST_COMMENT" else selected_column
source_data = pd.DataFrame(input_rows).rename(columns={"comment": source_column})
comparison = source_data.merge(pd.DataFrame(records), on="comment_id", validate="one_to_one")
output_file = ROOT / "outputs" / "00_preliminary" / "ECR_cost_comment_analysis.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Saved {len(comparison)} rows to {output_file.resolve()}")
display(comparison)